<div style="border:4px solid #164f73; border-radius:14px; padding:22px; background:#eef8ff; font-size:20px; line-height:1.55">

# 🚨 Judge Reproducibility Links: CADForge OpenEnv

**GitHub repo:** [https://github.com/sanjuhs/open-env-meta-final-hackathon](https://github.com/sanjuhs/open-env-meta-final-hackathon)  
**Run this notebook in Google Colab:** [https://colab.research.google.com/github/sanjuhs/open-env-meta-final-hackathon/blob/main/training/cadforge_openenv_training_colab.ipynb](https://colab.research.google.com/github/sanjuhs/open-env-meta-final-hackathon/blob/main/training/cadforge_openenv_training_colab.ipynb)  
**HF Space:** [https://huggingface.co/spaces/sanjuhs/cadforge-cadquery-openenv](https://huggingface.co/spaces/sanjuhs/cadforge-cadquery-openenv)  
**GitHub Gist: training scripts:** [https://gist.github.com/sanjuhs/10596f688e8b4560910a3b1b137bfeeb](https://gist.github.com/sanjuhs/10596f688e8b4560910a3b1b137bfeeb)  
**Raw training logs + evidence archive:** [https://huggingface.co/datasets/sanjuhs/cadforge-training-evidence](https://huggingface.co/datasets/sanjuhs/cadforge-training-evidence)

## What actually ran where

The **full SFT + GRPO training runs were executed on a RunPod H200**, not inside free Colab. The Colab notebook is the judge-runnable smoke path: it validates OpenEnv, loads the public dataset, runs the CadQuery reward backend, and launches tiny SFT/GRPO checks using the **same production scripts**.

Production training used distinct shell wrappers so CadQuery, Unsloth, TRL, model caches, and reward-backend dependencies stayed isolated and reproducible.

</div>

# CADForge OpenEnv Training Run Notebook

[Open this notebook in Google Colab](https://colab.research.google.com/github/sanjuhs/open-env-meta-final-hackathon/blob/main/training/cadforge_openenv_training_colab.ipynb) | [Notebook in the HF Space repo](https://huggingface.co/spaces/sanjuhs/cadforge-cadquery-openenv/blob/main/training/cadforge_openenv_training_colab.ipynb)

This is the public, judge-runnable training notebook for CADForge. It is intentionally split into two parts:

1. **Colab smoke run**: runs a tiny 3-step SFT training job and validates the CADForge reward pipeline so judges can verify the code path without renting an H200.
2. **Production run evidence**: links to the exact scripts, reports, model adapters, dataset, and full H200 commands used for the submitted runs.

The final submitted models were trained outside Colab on a RunPod H200 because Qwen3.5 9B + CADForge GRPO is too large for free Colab. The notebook does not pretend to rerun the full job; it proves the same training scripts and public dataset execute end-to-end on a tiny sample.


## What judges should check

- The repo clone and dependency install succeed.
- The OpenEnv app validates.
- The public HF dataset loads.
- The CADForge reward smoke test compiles and scores real CadQuery.
- The 3-step SFT smoke run calls `training/train_sft_unsloth.py`, the same script used for the production SFT runs.
- The optional GRPO cell calls `training/train_grpo_cadforge.py`, the same script used for strict build-gated GRPO.
- The final report cells show the committed metrics and generated artifacts from the full H200 training run.

## Public artifacts

- Project repo: https://github.com/sanjuhs/open-env-meta-final-hackathon
- Hugging Face Space repo: https://huggingface.co/spaces/sanjuhs/cadforge-cadquery-openenv

- GitHub repo, exact source: https://github.com/sanjuhs/open-env-meta-final-hackathon
- GitHub Gist: training scripts: https://gist.github.com/sanjuhs/10596f688e8b4560910a3b1b137bfeeb
- Raw training logs and evidence bundle: https://huggingface.co/datasets/sanjuhs/cadforge-training-evidence
- RunPod/H200 clarification: full 2B/9B SFT and GRPO runs were executed as distinct scripts on RunPod; this Colab is the public smoke rerun path.
- Training dataset: https://huggingface.co/datasets/sanjuhs/cadforge-cadquery-agentic-traces
- 2B SFT adapter: https://huggingface.co/sanjuhs/qwen35-2b-cadforge-sft-lora
- 2B GRPO adapter: https://huggingface.co/sanjuhs/qwen35-2b-cadforge-grpo-lora
- 9B SFT adapter: https://huggingface.co/sanjuhs/qwen35-9b-cadforge-sft-lora
- 9B GRPO adapter: https://huggingface.co/sanjuhs/qwen35-9b-cadforge-grpo-lora
- 9B strict build-gated GRPO adapter: https://huggingface.co/sanjuhs/qwen35-9b-cadforge-grpo-strict-build-lora


## 1. Configure

For a normal judge rerun, keep `RUN_SFT_SMOKE = True` and `RUN_GRPO_CADFORGE_SMOKE = False`.

The SFT smoke is the best Colab-friendly proof that the training script is real. The CADForge reward smoke separately proves that the OpenEnv/CadQuery reward backend is real. GRPO is left optional because even a tiny CADForge GRPO run can be slow or memory-heavy on Colab T4/L4 runtimes.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/sanjuhs/open-env-meta-final-hackathon.git"
REPO_DIR = Path("/content/open-env-meta-final-hackathon")

# Public dataset produced for CADForge.
DATASET = "sanjuhs/cadforge-cadquery-agentic-traces"

# Keep these small for public judge reruns.
RUN_SFT_SMOKE = True
RUN_GRPO_CADFORGE_SMOKE = False
SFT_SMOKE_STEPS = 3

# Make the flags visible to subprocess-backed shell cells.
os.environ["RUN_SFT_SMOKE"] = "1" if RUN_SFT_SMOKE else "0"
os.environ["RUN_GRPO_CADFORGE_SMOKE"] = "1" if RUN_GRPO_CADFORGE_SMOKE else "0"

# Optional. Only needed if you want to push adapters or access private repos.
os.environ.setdefault("HF_TOKEN", "")

BASE_2B = "unsloth/Qwen3.5-2B"
BASE_9B = "unsloth/Qwen3.5-9B"
print("Configured notebook smoke run")

import subprocess
import textwrap

def run_bash(script: str, check: bool = True) -> None:
    """Run a bash snippet from a normal Python Colab cell.

    Colab occasionally makes bash-magic cells awkward to save or rerun from a
    shared notebook. Keeping shell commands behind this helper makes every cell
    valid Python while still running the exact production commands.
    """
    env = os.environ.copy()
    env["PATH"] = f"{Path.home()}/.local/bin:" + env.get("PATH", "")
    subprocess.run(
        textwrap.dedent(script),
        shell=True,
        executable="/bin/bash",
        check=check,
        env=env,
    )


## 2. Clone and install

This uses `uv` to match the production RunPod environment. The install can take several minutes on a fresh Colab runtime.

In [ ]:
run_bash(r"""
set -euo pipefail

if [ ! -d /content/open-env-meta-final-hackathon/.git ]; then
  rm -rf /content/open-env-meta-final-hackathon
  git clone --depth 1 https://github.com/sanjuhs/open-env-meta-final-hackathon.git /content/open-env-meta-final-hackathon
else
  cd /content/open-env-meta-final-hackathon
  git fetch --depth 1 origin main
  git reset --hard origin/main
fi

cd /content/open-env-meta-final-hackathon
curl -LsSf https://astral.sh/uv/install.sh | sh || true
export PATH="$HOME/.local/bin:$PATH"

export UV_CACHE_DIR=/content/.uv-cache
export HF_HOME=/content/.cache/huggingface
export TORCH_HOME=/content/.cache/torch
export TRITON_CACHE_DIR=/content/.cache/triton
export VLLM_CACHE_ROOT=/content/.cache/vllm
export UV_LINK_MODE=copy
export HF_HUB_ENABLE_HF_TRANSFER=1
export HF_HUB_DISABLE_XET=1

uv sync --project experiment-2-cadforge
uv run --project experiment-2-cadforge python - <<'PY'
import numpy, trimesh, scipy, cadquery
print('CADForge project env OK:', numpy.__version__)
PY
""")


## 3. Validate the OpenEnv submission and inline reward backend

This validates the OpenEnv app, then evaluates a small pasted CadQuery program through the real CADForge reward environment.

The old root-level smoke command is kept as comments inside the cell because it can fail in Colab when `cadquery_env.py` runs outside the `experiment-2-cadforge` virtualenv. The working path is `uv run --project experiment-2-cadforge ...`, which gives the reward code access to `numpy`, `trimesh`, `scipy`, and `cadquery`.


In [ ]:
run_bash(r"""
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"

uv run --project experiment-2-cadforge openenv validate experiment-2-cadforge

# Kept for reference only. In Colab this can run from the root uv env and miss
# numpy/cadquery/trimesh, which causes ModuleNotFoundError inside cadquery_env.py.
# uv run training/smoke_cadforge_reward.py --reward-mode fast

cat > /tmp/cadforge_inline_candidate.py <<'CANDIDATEPY'
import cadquery as cq

# Tiny editable wall-mounted J-hook for the notebook smoke test.
# This is deliberately simple so judges can verify the CADForge reward path quickly.
plate_w = 46.0
plate_h = 72.0
plate_t = 6.0
hook_len = 58.0
hook_t = 12.0
hook_drop = 34.0
rib_t = 8.0
hole_d = 6.0

plate = cq.Workplane("XY").box(plate_w, plate_t, plate_h).translate((0, 0, plate_h / 2.0))

upper_hole = (
    cq.Workplane("XY")
    .circle(hole_d / 2.0)
    .extrude(plate_t + 2.0)
    .translate((0, -1.0, plate_h * 0.72))
)
lower_hole = (
    cq.Workplane("XY")
    .circle(hole_d / 2.0)
    .extrude(plate_t + 2.0)
    .translate((0, -1.0, plate_h * 0.28))
)
plate = plate.cut(upper_hole).cut(lower_hole)

arm = cq.Workplane("XY").box(hook_len, hook_t, hook_t).translate((hook_len / 2.0, 0, plate_h * 0.52))
tip = cq.Workplane("XY").box(hook_t, hook_t, hook_drop).translate((hook_len, 0, plate_h * 0.52 - hook_drop / 2.0))
rib = cq.Workplane("XY").box(hook_len * 0.65, rib_t, rib_t).translate((hook_len * 0.33, 0, plate_h * 0.38))

fixture = plate.union(arm).union(tip).union(rib).clean()
CANDIDATEPY

uv run --project experiment-2-cadforge python \
  experiment-2-cadforge/python_tools/cadquery_env.py evaluate \
  --code-file /tmp/cadforge_inline_candidate.py \
  --episode-id notebook-inline-smoke \
  --step-id pasted-cadquery-candidate \
  --task-spec wall_j_hook_120n \
  --task-prompt "Design a simple editable wall mounted J hook with a wall plate, hook arm, support rib, and screw holes." \
  --reward-mode fast
""")


## 4. Inspect the public training data

The production scripts download from the same public Hugging Face dataset if local JSONL files are absent.

In [ ]:
run_bash(r"""
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"

uv run --with datasets --with huggingface_hub python - <<'PY'
from datasets import load_dataset

repo = "sanjuhs/cadforge-cadquery-agentic-traces"
files = {
    "sft_train": "data/sft/prompt_to_cadquery_train.jsonl",
    "repair_train": "data/repair/cadquery_repair_train.jsonl",
    "rl_rollouts": "data/rl/cadquery_rollouts.jsonl",
}
for name, path in files.items():
    ds = load_dataset("json", data_files={name: f"hf://datasets/{repo}/{path}"}, split=name)
    print(name, len(ds), list(ds[0].keys()))
PY
""")


## 5. Three-step SFT smoke training

This runs the same SFT script used for the full runs, but with a tiny row limit and 3 optimizer steps. It is the Colab-friendly proof of the training entrypoint.

The production SFT runs used longer schedules and larger context on RunPod H200. Those commands and reports are shown below.

In [ ]:
run_bash(r"""
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"
export HF_HUB_DISABLE_XET=1

if [ "${RUN_SFT_SMOKE:-1}" != "1" ]; then
  echo "SFT smoke disabled in config."
  exit 0
fi

CUDA_READY=$(python - <<'PY'
import torch
print('1' if torch.cuda.is_available() else '0')
PY
)
echo "CUDA available: $CUDA_READY"
if [ "$CUDA_READY" != "1" ]; then
  echo "Skipping SFT smoke: switch Colab runtime to GPU, then rerun this cell."
  exit 0
fi

uv run training/train_sft_unsloth.py \
  --model unsloth/Qwen3.5-2B \
  --output-dir outputs/notebook-qwen35-2b-cadforge-sft-3step \
  --max-steps 3 \
  --limit-train-rows 12 \
  --limit-val-rows 4 \
  --max-seq-length 2048 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 1 \
  --eval-steps 3 \
  --save-steps 3 \
  --lora-r 16 \
  --lora-alpha 32 \
  --load-in-4bit \
  --run-name notebook-qwen35-2b-sft-3step
""")


## 6. Optional 3-step GRPO smoke

This calls the actual GRPO training script with strict build gating and the real CADForge reward backend. Leave it disabled for quick judging; enable it only on a strong GPU runtime.

In [ ]:
if not RUN_GRPO_CADFORGE_SMOKE:
    print("GRPO CADForge smoke disabled by default. Set RUN_GRPO_CADFORGE_SMOKE = True in the config cell to run it.")

In [ ]:
run_bash(r"""
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"
export HF_HUB_DISABLE_XET=1
export CADFORGE_PYTHON=/content/open-env-meta-final-hackathon/experiment-2-cadforge/.venv/bin/python

GRPO_READY=$(python - <<'PY'
import os, torch
run = os.environ.get('RUN_GRPO_CADFORGE_SMOKE', '').lower() in {'1', 'true', 'yes'}
print('1' if run and torch.cuda.is_available() else '0')
PY
)
echo "GRPO ready: $GRPO_READY"
if [ "$GRPO_READY" != "1" ]; then
  echo "Skipping optional GRPO smoke. Change RUN_GRPO_CADFORGE_SMOKE=1 in this cell and use a GPU runtime to run it."
  exit 0
fi

uv run training/train_grpo_cadforge.py \
  --model unsloth/Qwen3.5-2B \
  --adapter outputs/notebook-qwen35-2b-cadforge-sft-3step \
  --output-dir outputs/notebook-qwen35-2b-cadforge-grpo-3step \
  --reward-backend cadforge \
  --strict-build-gate \
  --cadforge-python "$CADFORGE_PYTHON" \
  --cadforge-reward-mode fast \
  --limit-prompts 2 \
  --max-steps 3 \
  --num-generations 2 \
  --max-prompt-length 2048 \
  --max-completion-length 1024 \
  --max-seq-length 4096 \
  --per-device-train-batch-size 2 \
  --gradient-accumulation-steps 1 \
  --learning-rate 5e-6 \
  --debug-completions-jsonl training/logs/notebook-grpo-3step-completions.jsonl \
  --run-name notebook-qwen35-2b-grpo-3step
""")


## 7. Exact production scripts

The full submitted run used the scripts below, not notebook-only code:

- `training/train_sft_unsloth.py`
- `training/train_grpo_cadforge.py`
- `training/evaluate_cadforge_model.py`
- `training/make_training_report.py`
- `training/cadforge_training_pipeline.sh`
- `training/run_strict_9b_grpo.sh`

The final strict GRPO run used build gating: failed CAD builds received negative reward, while successful builds unlocked dense topology, semantic, reference, and editability rewards.

In [ ]:
run_bash(r"""
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"

printf '\n--- SFT script CLI ---\n'
uv run training/train_sft_unsloth.py --help | sed -n '1,120p'

printf '\n--- GRPO script CLI ---\n'
uv run training/train_grpo_cadforge.py --help | sed -n '1,180p'

printf '\n--- Strict 9B production wrapper ---\n'
sed -n '1,220p' training/run_strict_9b_grpo.sh
""")


## 8. Production run evidence

These reports are committed in the public repo and mirrored in the Hugging Face Space repo. They are the evidence for the real H200 training run.

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

repo = Path('/content/open-env-meta-final-hackathon')
report_dir = repo / 'training/reports/qwen35-9b-grpo-strict-build-20260426-strict-build'

for image_name in ['grpo_reward_curve.png', 'grpo_code_health.png', 'grpo_error_breakdown.png']:
    path = report_dir / image_name
    if path.exists():
        display(Markdown(f'### {image_name}'))
        display(Image(filename=str(path)))
    else:
        print('Missing', path)

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

repo = Path('/content/open-env-meta-final-hackathon')
for rel in [
    'training/RUNPOD_SMOKE_REPORT.md',
    'training/reports/qwen35-9b-grpo-strict-build-20260426-strict-build/training_curve_report.md',
    'training/eval/qwen35-9b-cadforge-grpo-strict-build-20260426-strict-build/eval_report.md',
]:
    path = repo / rel
    display(Markdown(f'## {rel}'))
    if path.exists():
        display(Markdown(path.read_text()))
    else:
        print('Missing', path)

## 9. Summary of the submitted training run

- Qwen3.5-2B SFT: train loss `1.4480 -> 0.1658`, eval loss `0.4477 -> 0.2676`.
- Qwen3.5-9B SFT: train loss `2.6020 -> 0.1413`, eval loss `0.3650 -> 0.2398`.
- Qwen3.5-9B strict GRPO: `320` completions, `96` buildable completions, best CADForge score `0.9352`.
- Quick held-out eval after strict GRPO: `2/3` prompts built successfully.

The 3-step Colab run above is a reproducibility smoke test. The linked reports and adapters are the full training artifacts.

## RunPod H200 production scripts

The real training was launched on RunPod H200 through shell wrappers and Python scripts, then backed up to Hugging Face. Judges can inspect the public training-script Gist here: https://gist.github.com/sanjuhs/10596f688e8b4560910a3b1b137bfeeb

Raw logs and charts are here: https://huggingface.co/datasets/sanjuhs/cadforge-training-evidence
